# Generate Simulated Dynamic Multilayer Heterogeneous Data

This notebook creates MCCE-MHGNN-style dynamic multilayer heterogeneous snapshots. Historical years are used as input snapshots, and the target year stores `train_mask`, `valid_mask`, and `test_mask` on one target relation.

In [ ]:
import os
import random
import numpy as np
import torch
import dgl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR = 'data/simulated_dynamic_mcce'
os.makedirs(OUT_DIR, exist_ok=True)

YEARS = list(range(2015, 2021))
HISTORY_YEARS = YEARS[:-1]
TARGET_YEAR = YEARS[-1]
TARGET_ETYPE = ('author', 'coauthor', 'author')

NUM_AUTHORS = 3000
NUM_PAPERS = 9000
NUM_VENUES = 500
FEAT_DIM = 64
LATENT_DIM = 16

print('output:', OUT_DIR)

## Latent Factors

The target coauthor links are generated from stable author latent factors plus temporal drift. Paper and venue relations are correlated with author factors, so cross-layer metapath context is useful.

In [ ]:
author_z = torch.randn(NUM_AUTHORS, LATENT_DIM)
paper_z = torch.randn(NUM_PAPERS, LATENT_DIM)
venue_z = torch.randn(NUM_VENUES, LATENT_DIM)

author_feat_w = torch.randn(LATENT_DIM, FEAT_DIM) / LATENT_DIM ** 0.5
paper_feat_w = torch.randn(LATENT_DIM, FEAT_DIM) / LATENT_DIM ** 0.5
venue_feat_w = torch.randn(LATENT_DIM, FEAT_DIM) / LATENT_DIM ** 0.5

def make_feat(z, w, noise=0.1):
    x = z @ w + noise * torch.randn(z.shape[0], w.shape[1])
    return x.float()

def sample_author_paper(year_idx, edges_per_author=3):
    drift = 0.05 * year_idx * torch.randn_like(author_z)
    scores = (author_z + drift) @ paper_z.T
    src = torch.arange(NUM_AUTHORS).repeat_interleave(edges_per_author)
    top = torch.topk(scores, edges_per_author, dim=1).indices.reshape(-1)
    return src.long(), top.long()

def sample_paper_venue(year_idx):
    scores = paper_z @ venue_z.T + 0.03 * year_idx * torch.randn(NUM_PAPERS, NUM_VENUES)
    dst = torch.argmax(scores, dim=1)
    src = torch.arange(NUM_PAPERS)
    return src.long(), dst.long()

def sample_coauthor(year_idx, num_edges=18000):
    drift = 0.08 * year_idx * torch.randn_like(author_z)
    z = author_z + drift
    pairs_src = torch.randint(0, NUM_AUTHORS, (num_edges * 4,))
    pairs_dst = torch.randint(0, NUM_AUTHORS, (num_edges * 4,))
    keep = pairs_src != pairs_dst
    pairs_src, pairs_dst = pairs_src[keep], pairs_dst[keep]
    score = (z[pairs_src] * z[pairs_dst]).sum(dim=1)
    chosen = torch.topk(score, num_edges).indices
    return pairs_src[chosen].long(), pairs_dst[chosen].long()

def reverse(src, dst):
    return dst.clone(), src.clone()

In [ ]:
def build_graph(year):
    year_idx = year - YEARS[0]
    ap_src, ap_dst = sample_author_paper(year_idx)
    pa_src, pa_dst = reverse(ap_src, ap_dst)
    pv_src, pv_dst = sample_paper_venue(year_idx)
    vp_src, vp_dst = reverse(pv_src, pv_dst)
    aa_src, aa_dst = sample_coauthor(year_idx)

    data_dict = {
        ('author', 'coauthor', 'author'): (aa_src, aa_dst),
        ('author', 'author_to_paper', 'paper'): (ap_src, ap_dst),
        ('paper', 'paper_to_author', 'author'): (pa_src, pa_dst),
        ('paper', 'paper_to_venue', 'venue'): (pv_src, pv_dst),
        ('venue', 'venue_to_paper', 'paper'): (vp_src, vp_dst),
    }
    g = dgl.heterograph(data_dict, num_nodes_dict={
        'author': NUM_AUTHORS,
        'paper': NUM_PAPERS,
        'venue': NUM_VENUES,
    })
    g.nodes['author'].data['feat'] = make_feat(author_z, author_feat_w)
    g.nodes['paper'].data['feat'] = make_feat(paper_z, paper_feat_w)
    g.nodes['venue'].data['feat'] = make_feat(venue_z, venue_feat_w)
    g.nodes['author'].data['global_id'] = torch.arange(NUM_AUTHORS)
    g.nodes['paper'].data['global_id'] = torch.arange(NUM_PAPERS)
    g.nodes['venue'].data['global_id'] = torch.arange(NUM_VENUES)
    return g

graphs = {year: build_graph(year) for year in YEARS}
graphs[TARGET_YEAR]

## Attach Target-Year Split Masks

The masks are stored only on the target relation of the target-year graph.

In [ ]:
g_target = graphs[TARGET_YEAR]
num_edges = g_target.num_edges(TARGET_ETYPE)
perm = torch.randperm(num_edges)
n_train = int(num_edges * 0.7)
n_valid = int(num_edges * 0.1)

train_mask = torch.zeros(num_edges, dtype=torch.bool)
valid_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[perm[:n_train]] = True
valid_mask[perm[n_train:n_train+n_valid]] = True
test_mask[perm[n_train+n_valid:]] = True

g_target.edges[TARGET_ETYPE].data['train_mask'] = train_mask
g_target.edges[TARGET_ETYPE].data['valid_mask'] = valid_mask
g_target.edges[TARGET_ETYPE].data['test_mask'] = test_mask

print('target etype:', TARGET_ETYPE)
print('train/valid/test:', int(train_mask.sum()), int(valid_mask.sum()), int(test_mask.sum()))

In [ ]:
for year, g in graphs.items():
    path = os.path.join(OUT_DIR, f'graph_{year}.bin')
    dgl.save_graphs(path, [g])
    print(path, g)

## Training Example

In [ ]:
cmd = """
python -u Link_Prediction.py \
  --snapshot-bins data/simulated_dynamic_mcce/graph_2015.bin,data/simulated_dynamic_mcce/graph_2016.bin,data/simulated_dynamic_mcce/graph_2017.bin,data/simulated_dynamic_mcce/graph_2018.bin,data/simulated_dynamic_mcce/graph_2019.bin \
  --target-bin data/simulated_dynamic_mcce/graph_2020.bin \
  --target-etype author:coauthor:author \
  --metapaths "author:author_to_paper:paper>paper:paper_to_author:author,author:author_to_paper:paper>paper:paper_to_venue:venue>venue:venue_to_paper:paper>paper:paper_to_author:author" \
  --context-encoder gcn \
  --metapath-fusion conv \
  --fusion-mode both \
  --temporal-model gru \
  --predictor distmult \
  --hidden-dim 128 \
  --epochs 500 \
  --log-every 10 \
  --output-dir outputs \
  --undirected
"""
print(cmd)